# A3 — Domain-format stress test and one fine-tuning trial

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys
from google.colab import drive, userdata

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/KSE_AI301_A3')
SOURCE = ROOT/'domain_shift_current_source'
manifest = json.loads((SOURCE/'source_manifest.json').read_text())
expected_source_id = '2077747a556e250be4b5107c75c1bfb2231054e718c4840676fd323110514a07'
assert manifest['source_id'] == expected_source_id, 'Unreviewed source manifest'
check = hashlib.sha256(json.dumps(manifest['files'], sort_keys=True,
    separators=(',', ':')).encode()).hexdigest()
assert check == expected_source_id, 'Source manifest content changed'
REPO = Path('/content/a3_domain_shift_repo')
for relative, expected in manifest['files'].items():
    actual_source = SOURCE/relative
    assert actual_source.is_file(), 'Missing source: ' + str(actual_source)
    assert hashlib.sha256(actual_source.read_bytes()).hexdigest() == expected, 'Source changed: ' + relative
    target = REPO/relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(actual_source, target)
    assert hashlib.sha256(target.read_bytes()).hexdigest() == expected
print('Verified source tree:', len(manifest['files']), 'files, SHA256:', expected_source_id)
os.environ['A3_SOURCE_SHA'] = expected_source_id
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(REPO/'requirements-domain-shift.txt')], check=True)
sys.path.insert(0, str(REPO))


In [ ]:
import torch
assert torch.cuda.is_available(), 'Choose GPU runtime'
key = os.environ.get('WANDB_API_KEY') or userdata.get('WANDB_API_KEY')
assert key, 'Add WANDB_API_KEY to Colab Secrets and allow notebook access'
os.environ['WANDB_API_KEY'] = key
os.environ['WANDB_MODE'] = 'online'
os.environ['WANDB_PROJECT'] = 'ai301-a3'
os.environ['WANDB_ENTITY'] = 'dtkach-kse'
import wandb
assert wandb.login(), 'W&B login failed'
list(wandb.Api().runs('dtkach-kse/ai301-a3', per_page=1))
print('W&B connected; GPU:', torch.cuda.get_device_name(0))


In [ ]:
from part2_data import file_hash, read_split
from part2 import copy_file

spec_path = REPO/'part2_split.json'
spec = read_split(spec_path)
DATA = Path('/content/a3_domain_shift_data')
DATA.mkdir(parents=True, exist_ok=True)
for name, expected in spec['files'].items():
    local = DATA/name
    if not local.exists() or file_hash(local) != expected:
        source = ROOT/'data'/name
        assert source.is_file(), 'Required source CSV missing in Drive: ' + str(source)
        copy_file(source, local)
    assert file_hash(local) == expected, 'Source data hash mismatch: ' + name

checkpoint = ROOT/'results/word_representations_v1/ngrams/main/best.pt'
expected_checkpoint = 'ed3c9e14bba969a129d3f70e670e05b968815bb93e4586fb1161cc0f7afeaa29'
assert checkpoint.is_file() and file_hash(checkpoint) == expected_checkpoint, 'Selected ngrams checkpoint missing or changed'
OUT = ROOT/'results/domain_shift_reproduction'
OUT.mkdir(parents=True, exist_ok=True)
print('Verified frozen CSVs, split and checkpoint; outputs:', OUT)


In [ ]:
from part2_data import load_data
from experiments.av_diagnostics import run as av_run
frames, frozen_split = load_data(DATA, spec_path)
av = av_run(frames, frozen_split, OUT)
print('Domain separability by feature family (OOF ROC AUC):', av)
with wandb.init(project='ai301-a3', entity='dtkach-kse',
                name='av_length_format_ablation', job_type='domain_shift_diagnostic') as diagnostic_run:
    diagnostic_run.log({'av/'+name: score['oof_auc'] for name, score in av.items()})


In [ ]:
from collections import deque
import subprocess, sys
args = [sys.executable, '-u', '-m', 'experiments.format_finetune',
        '--data-dir', str(DATA), '--split', str(spec_path),
        '--checkpoint', str(checkpoint), '--checkpoint-sha256', expected_checkpoint,
        '--output-dir', str(OUT), '--project', 'ai301-a3', '--entity', 'dtkach-kse']
recent = deque(maxlen=40)
log_path = OUT/'finetune_console.log'
with log_path.open('a', encoding='utf-8') as logfile:
    logfile.write('\n=== Fine-tuning invocation, pinned source: ' + expected_source_id + ' ===\n')
    with subprocess.Popen(args, cwd=REPO, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
            recent.append(line)
            logfile.write(line)
        exit_code = process.wait()
if exit_code:
    print('\n--- TRAINER ERROR: last 40 output lines ---')
    print(''.join(recent))
    print('Full log:', log_path)
    raise RuntimeError(f'Fine-tuning failed (exit={exit_code}); see the traceback above')
print('Fine-tuning completed successfully; full log:', log_path)


In [ ]:
result = json.loads((OUT/'decision.json').read_text())
print('Pinned source SHA256:', expected_source_id)
print('Status:', result['status'], '| Accepted:', result['accepted'])
print('Gates:', result['gates'])
print('Metric changes:', result['gains'])
print('Stress paired-bootstrap 95% CI:', result['paired_stress_delta_f1_95ci'])
print('Candidate SHA256:', result['checkpoint_sha256'])
print('Files:', OUT/'decision.json', OUT/'history.csv', OUT/'candidate.pt')